# Azure Stream Data Product — Pre-Development Framework (v1)

## Purpose

Define the guarantees and boundaries of the Azure streaming system before execution.

This document declares:

- What the system guarantees
- What it explicitly does not attempt
- What state is maintained
- What recovery behavior exists

This is a rulebook.
It does not describe implementation details.

---

## System Overview

Execution flow:

Generator → Event Hubs → Spark Structured Streaming → Delta Lake

The streaming processor performs:

1. Continuous micro-batch ingestion
2. Contract-based validation
3. Watermark-based deduplication
4. Incremental metric updates
5. Delta Lake publication
6. Checkpoint-based recovery

---

## Scope

### In Scope

- Micro-batch streaming (30-second trigger)
- Azure Event Hubs ingestion
- Spark Structured Streaming
- Delta Lake storage
- Exactly three published metrics
- Watermark-based deduplication
- At-least-once processing with idempotency

### Out of Scope

- Exactly-once end-to-end guarantees
- Dedicated monitoring tables
- Rejected-events persistence
- Batch recomputation pipelines
- Complex event processing
- Multi-stream joins
- Multi-currency normalization
- Real-time alerting
- Performance benchmarking
- Custom state store implementation

This system intentionally focuses on core streaming patterns.

---

## Core Guarantees

### 1. Contract-Based Validation

Input events must satisfy:

- Required schema fields
- Correct data types
- Non-null required columns
  (`event_id, user_id, event_time, ingest_time, event_type, amount, currency, channel`)
- Valid `event_type` values
  (`deposit_completed`, `withdrawal_completed`)
- Valid `currency` values
  (`GBP`)
- Positive `amount` values

Invalid events:
- Are filtered out of metric computation
- Are not written to a rejected-events table in v1

No silent schema coercion.

---

### 2. Micro-Batch Processing

Events are processed in micro-batches:

```
Trigger interval: 30 seconds
Expected throughput: ~100 events/sec
```

Each micro-batch:

- Reads new offsets from Event Hubs
- Parses JSON payload
- Validates contract
- Applies deduplication
- Updates metrics
- Commits Delta transactions
- Advances checkpoint

---

### 3. Deduplication Model

Deduplication key:

```
event_id
```

Implementation approach:

- Watermark-based deduplication via Spark state store
- `withWatermark(event_time, horizon)` + `dropDuplicates(["event_id"])`

Guarantee:

- Duplicate events within the watermark horizon are removed
- Idempotency is achieved within that horizon
- Duplicates outside the watermark window may be reprocessed

Processing semantics:

- At-least-once ingestion
- Application-level idempotency via event_id

---

### 4. Stateful Aggregations

State is maintained in two locations:

**Spark State Store**
- Used for watermark-based deduplication

**Delta Tables**
- Used for metric persistence (running totals)
- Updated via atomic MERGE operations

No custom state stores are implemented.

---

## Published Metrics (v1)

Exactly three metrics are produced.

Any additional metric requires a version bump.

---

### M1 — Net Flow

**Table:** `metrics/net_flow/`

**Grain:** Single row — global cumulative totals

**Update pattern:**
Atomic Delta MERGE on a constant key. ACID-safe. No read-overwrite.

**Schema:**

```
total_deposits:              double
total_withdrawals:           double
net_flow:                    double
deposit_count:               bigint
withdrawal_count:            bigint
avg_deposit:                 double
avg_withdrawal:              double
deposit_to_withdrawal_ratio: double
updated_at:                  timestamp
```

Guarantee:
- Single source of truth for global flow
- ACID-safe cumulative updates
- No race conditions under retry

---

### M2 — User Metrics

**Table:** `metrics/user_metrics/`

**Grain:** One row per `user_id`

**Update pattern:**
Delta MERGE (upsert) — running totals accumulated per user across all batches

**Schema:**

```
user_id:           string
total_deposits:    double
total_withdrawals: double
deposit_count:     bigint
withdrawal_count:  bigint
first_seen:        timestamp
last_seen:         timestamp
updated_at:        timestamp
```

Guarantee:
- Incremental per-user accumulation
- First and last activity timestamps maintained
- Idempotent within watermark window

---

### M3 — Channel Distribution

**Table:** `metrics/channel_distribution/`

**Grain:** One row per `(channel, event_type)`

**Update pattern:**
Delta MERGE (upsert) — running totals per channel and event type

**Schema:**

```
channel:      string
event_type:   string
event_count:  bigint
total_amount: double
updated_at:   timestamp
```

Guarantee:
- Running totals by channel and event type
- Incremental updates per micro-batch

---

## Curated Layer (Optional)

Valid events may be written to:

```
curated/transaction_events/
```

This layer is optional and disabled by default.

Purpose:

- Audit history of valid events
- Potential source for future recomputation
- Contract-validated event log

---

## Checkpoint Strategy

Checkpoints stored at:

```
checkpoints/
```

Contains:

- Event Hub offsets
- Watermark state
- Deduplication state
- Batch metadata

Recovery behavior:

- On failure, restart resumes from last committed checkpoint
- At most one micro-batch may be replayed
- Deduplication prevents double-counting within watermark horizon

---

## Failure Handling

If a micro-batch fails:

- Stream stops
- Checkpoint remains intact
- No partial metric publication occurs
- Restart resumes safely

Delta transactions are atomic.

---

## Update Cadence

```
Trigger interval: 30 seconds
Metric refresh:   every micro-batch
```

Metrics are near-real-time (≤30-second latency).

---

## Generator Characteristics

Synthetic event generator:

- 100 events per second
- ~500 active users
- Channels: `web`, `mobile`, `api`
- Currency: `GBP`
- Amount range: £5 – £500
- Event types: `deposit_completed`, `withdrawal_completed`

This dataset is controlled simulation for streaming pattern validation.

---

## Design Trade-offs

### Micro-batching over Continuous Processing

Chosen for:

- Operational simplicity
- Atomic Delta commits
- Predictable cost
- Easier debugging

Trade-off:
- 30-second latency

---

### At-Least-Once vs Exactly-Once

Chosen for:

- Simpler architecture
- Industry-standard analytics pattern
- Idempotency through watermark deduplication

Trade-off:
- Deduplication bound by watermark horizon

---

### Three Metrics Only

Intentional constraint:

- M1 demonstrates single-row ACID merge
- M2 demonstrates entity upserts
- M3 demonstrates multi-grain aggregation

Depth over breadth.

---

## Definition of Healthy Stream

Stream is healthy when:

- Micro-batches complete consistently
- Checkpoints advance
- Delta tables update every trigger interval
- No sustained lag accumulation in Event Hubs
- Processing time remains below trigger interval

---

## What This Project Demonstrates

- Event Hubs ingestion
- Structured Streaming micro-batching
- Contract validation in streaming context
- Watermark-based deduplication
- Delta Lake ACID merges
- Stateful streaming with checkpoint recovery
- Incremental metric design
- Trade-off documentation